# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:

1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**

1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


## 0. Setup and token

**Method, step by step:**

1. **Installs** DuckDB and the Hugging Face client (a no-op if they are already there).
2. **Finds the token** in this order: an environment variable, a Colab Secret, and only as a last resort a hidden prompt. The token is held in memory and never printed or written into a cell.
3. **Loads the tools** and defines `show()` and `show_table()`, so every result is a titled table in the same layout, plus the chart theme used by the one chart below.


In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import getpass
import os


def read_colab_secret(name: str) -> str | None:
    """Return the Colab Secret called `name`, or None when it cannot be read.

    Outside Colab the import fails. On Colab the secret can be missing, not shared with this
    notebook, or the request can time out. Each case returns None so the caller can ask for the token.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException) as error:
        print(f'Colab secret {name!r} unavailable ({type(error).__name__}); falling back to a prompt.')
        return None


# Token order: env var -> Colab Secret -> prompt (last resort).
HF_TOKEN = os.environ.get('HF_TOKEN') or read_colab_secret('HF_TOKEN')
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [ ]:
import os
import tempfile
import urllib.request

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib import font_manager

pd.set_option("display.width", 160)
NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))

# Emphasis theme: cividis navy for the story, grays for what should recede.
INK, MUTED, FAINT, GRID, SURFACE = "#111111", "#5f5e5a", "#9a9893", "#eeede9", "#ffffff"
NAVY, GRAY, PALE = mpl.colormaps["cividis"](0.05), "#9a9893", "#d4d2cc"


def load_inter() -> str:
    """Register the Inter font for this session and return the font family to draw with.

    The 400 and 600 weights are cached in the system temp folder, never in the project.
    If the download is unavailable, the chart falls back to matplotlib's default font,
    so the notebook still runs offline.
    """
    cache = os.path.join(tempfile.gettempdir(), "inter-font")
    os.makedirs(cache, exist_ok=True)
    try:
        for weight in (400, 600):
            path = os.path.join(cache, f"inter-{weight}.ttf")
            if not os.path.exists(path):
                url = f"https://cdn.jsdelivr.net/fontsource/fonts/inter@latest/latin-{weight}-normal.ttf"
                urllib.request.urlretrieve(url, path)
            font_manager.fontManager.addfont(path)
    except OSError:
        return "DejaVu Sans"
    return "Inter"


plt.rcParams.update({"font.family": load_inter()})

## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


**Method:** this cell opens a DuckDB session, registers the token as a secret so every `hf://` read is authenticated, and names the five tables. It then counts the rows of each one. A `COUNT(*)` over Parquet only needs the files' metadata, so it finishes in seconds even for the 79-million-row daily table.


In [ ]:
import duckdb

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
try:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
except duckdb.Error as error:
    # Re-raise without the token so it cannot end up in a saved error message.
    raise RuntimeError("Could not register the Hugging Face token: " + str(error).replace(HF_TOKEN, "<token>")) from None

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
HOLDS = {
    'dim_clients': "one row per client, with its data-access profile and start dates",
    'dim_content': "one row per content item and its descriptive fields",
    'fact_daily': "daily search performance for every content item",
    'fact_daily_sample': "a small sample of the daily table, for quick tests",
    'fact_query_90d': "aggregated query-level signals over 90 days, never raw queries",
}

counts = {name: con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0] for name, src in TABLES.items()}
show_table("The five tables in the release", pd.DataFrame({
    "What it holds": pd.Series(HOLDS),
    "Rows": pd.Series(counts).map("{:,}".format),
}).rename_axis("Table"))

That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an _unbalanced panel_). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


**Method, step by step:**

1. **Reads the release window** with `MIN` and `MAX` over the daily table, which Parquet answers from file statistics, so nothing is scanned.
2. **Pulls the client list.** `dim_clients` is tiny, so it comes straight into pandas.
3. **Measures each client's history** as the months between its first Search Console day and the last day of the release, and groups the clients by data-access profile.


In [ ]:
window = con.sql(f"SELECT MIN(report_date) AS first_day, MAX(report_date) AS last_day FROM {TABLES['fact_daily']}").df()
first_day, last_day = pd.Timestamp(window.loc[0, "first_day"]), pd.Timestamp(window.loc[0, "last_day"])

clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
clients["history_months"] = (last_day - clients["gsc_data_start"]).dt.days / 30.44

by_profile = clients.groupby("access_profile", dropna=False).agg(
    clients=("client_hash_id", "size"),
    with_ga4=("ga4_data_start", lambda s: s.notna().sum()),
    median_months=("history_months", "median"),
    longest_months=("history_months", "max"),
    twelve_plus=("history_months", lambda s: (s >= 12).sum()),
).sort_values("clients", ascending=False)

show("The release window", {
    "first day in the daily table": f"{first_day:%Y-%m-%d}",
    "last day in the daily table": f"{last_day:%Y-%m-%d}",
    "clients in the release": len(clients),
    "clients with 12 or more months of Search Console history": f"{(clients['history_months'] >= 12).sum()}",
})
show_table("Clients by data-access profile", pd.DataFrame({
    "Clients": by_profile["clients"],
    "With GA4 data": by_profile["with_ga4"],
    "Median history (months)": by_profile["median_months"].map("{:.1f}".format),
    "Longest history (months)": by_profile["longest_months"].map("{:.1f}".format),
    "12+ months": by_profile["twelve_plus"],
}).rename_axis("Access profile"))

## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


**Method:** this cell aggregates the daily table inside DuckDB into one row per content item. For each item it adds up impressions and clicks over the last 30 days and over the 30 days before that, and averages the last 30 days' position. The `HAVING` line keeps only items with at least 100 impressions in the previous 30 days, so a decline is measured against a base big enough to mean something. Only this small result comes back into pandas. If the cell runs past about ten minutes, its first line is the one switch to change: point `DAILY` at the sample table, and say so in the Finding below.


In [ ]:
# If this cell runs past about 10 minutes, change the next line to TABLES['fact_daily_sample'] and run it again.
DAILY = TABLES['fact_daily']

features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {DAILY}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {DAILY} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

show("Content items with enough history", {
    "items kept (100+ impressions in the previous 30 days)": f"{len(features):,}",
    "clients they belong to": features["client_hash_id"].nunique(),
    "median impressions, previous 30 days": f"{features['imp_prev30'].median():,.0f}",
    "median impressions, last 30 days": f"{features['imp_last30'].median():,.0f}",
    "items whose last 30 days fell below 80% of the previous 30": f"{(features['imp_last30'] < 0.8 * features['imp_prev30']).mean():.1%}",
})

## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


**Method:** this cell summarises the query table, which has many rows per content item, down to one row per item: how many distinct queries it shows for, how much of its traffic comes from rare or anonymized queries, and how much of it the single top query carries. It then joins that onto the feature table. Only these aggregate shares are used; no query text is read or shown.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')

show_table("Joining the query signals onto the feature table", pd.DataFrame({
    "Value": {
        "feature rows": f"{len(features):,}",
        "rows after the join": f"{len(data):,}",
        "items with query signals": f"{data['visible_queries'].notna().sum():,} ({data['visible_queries'].notna().mean():.1%})",
        "items without query signals": f"{data['visible_queries'].isna().sum():,} ({data['visible_queries'].isna().mean():.1%})",
    }
}).rename_axis("Measure"))

## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: _did impressions decline by more than 20% month-over-month?_ — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


**Method, step by step:**

1. **Defines the label:** an item is `is_declining` when its last 30 days of impressions are under 80% of the previous 30.
2. **Keeps the items that have every feature,** so no row is filled with a guess.
3. **Holds out a random quarter** of those rows, stratified so train and test have the same share of declining items.
4. **Fits a 200-tree random forest** on the training rows and scores it on the held-out rows.
5. **Compares it with the dumb baseline** of always predicting the majority class, and shows which features carry the signal.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

report = classification_report(y_te, model.predict(X_te), digits=3, output_dict=True)
random_split = {
    "accuracy": report["accuracy"],
    "auc": roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]),
    "precision": report["1"]["precision"],
    "recall": report["1"]["recall"],
}
majority = max(y_te.mean(), 1 - y_te.mean())

show_table("The first model, scored on a random quarter of the rows", pd.DataFrame({
    "Value": {
        "rows used (all five features present)": f"{len(model_data):,}",
        "rows held out to test": f"{len(X_te):,}",
        "share of declining items": f"{y_te.mean():.1%}",
        "accuracy of always predicting the majority class": f"{majority:.3f}",
        "model accuracy": f"{random_split['accuracy']:.3f}",
        "model ROC-AUC": f"{random_split['auc']:.3f}",
        "precision on declining items": f"{random_split['precision']:.3f}",
        "recall on declining items": f"{random_split['recall']:.3f}",
    }
}).rename_axis("Measure"))
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
show_table("Which features carry the signal", pd.DataFrame({"Importance": importance.map("{:.3f}".format)}).rename_axis("Feature"))

Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — _does it generalize across clients?_ — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

**Method, step by step:**

1. **Splits by client** five times with `GroupShuffleSplit`, holding out a quarter of the clients each time, so every test item comes from a site the model never saw.
2. **Fits the same random forest** with the same settings on the training clients only, and scores the held-out clients.
3. **Compares** accuracy, ROC-AUC, precision and recall against the random split above, and reports the lowest and highest of the five client splits so the spread is visible.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

N_SPLITS = 5
groups = model_data['client_hash_id']
splitter = GroupShuffleSplit(n_splits=N_SPLITS, test_size=0.25, random_state=42)

rows = []
for train_idx, test_idx in splitter.split(X, y, groups=groups):
    held = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = held.predict_proba(X.iloc[test_idx])[:, 1]
    rep = classification_report(y.iloc[test_idx], (scores > 0.5).astype(int), output_dict=True, zero_division=0)
    rows.append({
        "test_items": len(test_idx), "test_clients": groups.iloc[test_idx].nunique(), "base_rate": y.iloc[test_idx].mean(),
        "accuracy": rep["accuracy"], "auc": roc_auc_score(y.iloc[test_idx], scores),
        "precision": rep["1"]["precision"], "recall": rep["1"]["recall"],
    })
grouped = pd.DataFrame(rows)

METRICS = ["accuracy", "auc", "precision", "recall"]
NAMES = {"accuracy": "Accuracy", "auc": "ROC-AUC", "precision": "Precision on declining items", "recall": "Recall on declining items"}
show_table("Random split against client holdout", pd.DataFrame({
    "Random split": [f"{random_split[m]:.3f}" for m in METRICS],
    "Client holdout, mean of 5": [f"{grouped[m].mean():.3f}" for m in METRICS],
    "Lowest of 5": [f"{grouped[m].min():.3f}" for m in METRICS],
    "Highest of 5": [f"{grouped[m].max():.3f}" for m in METRICS],
}, index=pd.Index([NAMES[m] for m in METRICS], name="Measure")))
show("The five client splits", {
    "typical test group": f"{grouped['test_items'].median():,.0f} items from {grouped['test_clients'].median():.0f} clients",
    "base rate of declining items across the five test groups": f"{grouped['base_rate'].min():.1%} to {grouped['base_rate'].max():.1%}",
})

In [ ]:
# Chart: each score on a random split against its mean on held-out clients.
ROWS = [("auc", "ROC-AUC", NAVY), ("precision", "Precision", GRAY), ("recall", "Recall", GRAY), ("accuracy", "Accuracy", GRAY)]
lo = min(min(random_split[m], grouped[m].mean()) for m, _, _ in ROWS)
hi = max(max(random_split[m], grouped[m].mean()) for m, _, _ in ROWS)
pad = (hi - lo) * 0.25

fig, ax = plt.subplots(figsize=(10, 4.6), dpi=120, facecolor=SURFACE)
fig.subplots_adjust(left=0.16, right=0.95, top=0.80, bottom=0.12)
ax.set_facecolor(SURFACE)
for row, (metric, label, color) in enumerate(reversed(ROWS)):
    start, end = random_split[metric], grouped[metric].mean()
    story = color == NAVY
    ax.annotate("", xy=(end, row), xytext=(start, row), zorder=2,
                arrowprops={"arrowstyle": "-|>", "color": color, "lw": 2.8 if story else 2.0, "mutation_scale": 15,
                            "shrinkA": 8, "shrinkB": 8})
    ax.scatter(start, row, s=160, facecolor=SURFACE, edgecolor=color, linewidth=2.4, zorder=3)
    ax.scatter(end, row, s=160, facecolor=color, edgecolor=SURFACE, linewidth=2, zorder=3)
    for x, other in ((start, end), (end, start)):
        side = 1 if x >= other else -1
        ax.text(x + side * pad * 0.12, row, f"{x:.3f}", ha="left" if side > 0 else "right", va="center",
                color=INK if story else MUTED, fontsize=12.5 if story else 11.5, fontweight="semibold" if story else "regular")
    ax.text(lo - pad * 0.55, row, label, ha="right", va="center", color=INK if story else MUTED, fontsize=12.5,
            fontweight="semibold" if story else "regular")
ax.set_xlim(lo - pad * 0.5, hi + pad)
ax.set_ylim(-0.6, len(ROWS) - 0.4)
ax.set_yticks([])
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_axisbelow(True)
ax.tick_params(axis="x", length=0, colors=FAINT, labelsize=10, pad=8)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.text(0.5, 0.90, f"ROC-AUC moves from {random_split['auc']:.3f} to {grouped['auc'].mean():.3f} on unseen clients",
         ha="center", color=INK, fontsize=18, fontweight="semibold")
plt.show()

## Limitations

- **The query table may overlap the label window.** `fact_query_90d` is named for a 90-day window. If that window ends at the release date, it includes the same last 30 days the label is built from, so the query-mix features may know a little about the outcome. Treat their importance with that in mind.
- **The label depends on a feature.** `is_declining` is `imp_last30 < 0.8 * imp_prev30`, and `imp_prev30` is both a feature and the denominator of the label, so part of what the model learns is the label's own arithmetic.
- **Five splits are not five independent tests.** They are drawn from the same clients, so their spread understates how much a different set of clients could change the score.
- **Decline is not a decision.** A drop of a fifth in impressions can be a season, an algorithm update or a lost ranking, and nothing here says what a team should do about it.
- **Only five features.** This is a first honest model, not a tuned one; it shows the workflow rather than the best result the release can give.


## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```

Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.